In [59]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [60]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

In [61]:
wiki.name

'wikipedia'

In [62]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents =RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents, HuggingFaceEmbeddings())
retriever = vectordb.as_retriever()
retriever

C:\Users\NATHAN\AppData\Local\Temp\ipykernel_17292\1263200720.py:9: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  vectordb = FAISS.from_documents(documents, HuggingFaceEmbeddings())
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 948.31it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002206E43F4C0>, search_kwargs={})

In [63]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever,"langsmith_search",
"Search for information about LangSmith. For any questions about LangSmith, you must use this tool!")

In [64]:
retriever_tool.name

'langsmith_search'

In [65]:
## Arxiv
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [66]:
tools = [wiki,arxiv,retriever_tool]

In [67]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\NATHAN\\Langchain\\venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='langsmith_search', description='Search for information about LangSmith. For any questions about LangSmith, you must use this tool!', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000002206DE48820>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000002206C9F2320>)]

In [68]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1")

In [69]:
#from langchain_core.prompts import ChatPromptTemplate
#from langchain_core.output_parsers import StrOutputParser

#prompt = ChatPromptTemplate.from_messages([
#  ("system", "You are an assistant for question-answering tasks. Use the retrieved context to answer. If you don't know, say you don't know. Use three sentences maximum and keep the answer concise."),
#  ("human", "Question: {question}\nContext: {context}\nAnswer:")
#])

#output_parser = StrOutputParser()


In [70]:
### Agents
from langchain.agents import create_agent
agent = create_agent(model=llm, tools=tools, system_prompt="You are an assistant for question-answering tasks. Use the retrieved context to answer. If you don't know, say you don't know. Use three sentences maximum and keep the answer concise.")


In [ ]:
## Running the Agent
result=agent.invoke(
    {"messages": [{"role":"user",
    "content": "Explain about langsmith"}]}
)

In [78]:
result['messages'][-1].content

'LangSmith is a tool for developing, debugging, and deploying LLM (Large Language Model) applications. It provides features such as observability, evaluation, prompt engineering, deployment, and platform setup in one integrated workflow. LangSmith helps you create reliable AI systems by tracing requests, evaluating outputs, testing prompts, and managing deployments locally or in the cloud.'